# Cost–Time Plan Optimization
### Generate the plan pipelines

In [5]:
import os, re, json, glob, filecmp
import pandas as pd
import yaml
import jinja2

TEMPLATES = "../2. Cost-Time Model Accuracy/pipeline"
DATA, OUT = "data", "pipeline"
PLANS = None
QUERIES = None
WITH_NO_BF = True

qnum = lambda q: int("".join(c for c in q if c.isdigit()) or 0)
PL = pd.read_csv(f"{DATA}/plans.csv")
todo = PL[(PL.plan == PL.run_as)  & (PL.plan.isin(PLANS) if PLANS else True) & (PL["query"].isin(QUERIES) if QUERIES else True)]

print(f"{len(todo)} distinct plans over {todo['query'].nunique()} queries")
todo.groupby("plan").size().to_frame("queries").T

145 distinct plans over 19 queries


plan,default,front_02,front_03,front_04,front_05,knee,min_C,min_T
queries,19,18,17,18,18,19,18,18


## Render a template and apply a plan

In [6]:
def as_list(x):
    if x is None or x == {} or (isinstance(x, str) and x.strip().lower() in ("", "none")):   # q14 writes `none`
        return []
    return [x] if isinstance(x, str) else list(x)


def render(q, enable_bf):
    text = open(f"{TEMPLATES}/{q}.yml.j2").read().replace("$enable_bf", str(int(enable_bf)))
    head = yaml.safe_load(text.split("\nsteps:")[0])["variables"]
    env = {k: v for k, v in head.items() if "{{" not in str(v)}
    return yaml.safe_load(jinja2.Template(text).render(**env))


def apply_plan(raw, plan):
    steps = raw["steps"]
    by_name = {s["name"].lower(): s for s in steps}
    for m in plan["bf_off"]:
        merge = by_name[m.lower()]
        for b in as_list(merge.get("from_previous_task")):
            for k in ("bf_column", "bf_path", "est_elements"):
                by_name[b.lower()].get("params", {}).pop(k, None)
        for s in steps:
            deps = as_list(s.get("from_previous_task"))
            if merge["name"] in deps:
                rest = [x for x in deps if x != merge["name"]]
                if rest:
                    s["from_previous_task"] = rest if len(rest) > 1 else rest[0]
                else:
                    s.pop("from_previous_task")
                for k in ("filter_by_bf", "bf_column"):
                    s.get("params", {}).pop(k, None)
        steps.remove(merge)
    for name, mb in plan["max_size_mb"].items():
        by_name[name.lower()]["max_size_mb"] = mb
    return raw


def check(raw, plan=None):
    steps = {s["name"]: s for s in raw["steps"]}
    names = {n.lower() for n in steps}
    if plan is None:
        assert not any(s["func"] == "merge_bf" for s in steps.values()), "a merge_bf step is left"
    else:
        assert not names & {m.lower() for m in plan["bf_off"]}, "a removed merge_bf step is still there"
        for n, mb in plan["max_size_mb"].items():
            assert next(s for s in steps.values() if s["name"].lower() == n.lower())["max_size_mb"] == mb, f"max_size_mb of {n}"
    for s in steps.values():
        for d in as_list(s.get("from_previous_task")):
            assert d in steps, f"{s['name']} depends on missing step {d}"
    return len(steps)


def write(raw, plan_name, q):
    path = os.path.join(OUT, plan_name, f"{q}.yml.j2")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        yaml.safe_dump(raw, f, sort_keys=False, allow_unicode=True)
    return path

## Write the pipelines

In [7]:
rows = []
for r in todo.sort_values(["query", "plan"], key=lambda s: s.map(qnum) if s.name == "query" else s).itertuples():
    plan = {"bf_off": str(r.bf_off).split() if isinstance(r.bf_off, str) else [],
            "max_size_mb": json.loads(r.max_size_mb)}
    raw = apply_plan(render(r.query, True), plan)
    rows.append({"query": r.query, "plan": r.plan, "steps": check(raw, plan), "BFs off": len(plan["bf_off"]),
                 "stages resized": len(plan["max_size_mb"]), "path": write(raw, r.plan, r.query)})
if WITH_NO_BF:
    for q in sorted(todo["query"].unique(), key=qnum):
        raw = render(q, False)
        rows.append({"query": q, "plan": "no_bf", "steps": check(raw), "BFs off": None, "stages resized": 0,
                     "path": write(raw, "no_bf", q)})
W = pd.DataFrame(rows)
print(f"{len(W)} pipelines -> {OUT}/")
W.pivot_table(index="query", columns="plan", values="steps", aggfunc="first").reindex(
    sorted(W["query"].unique(), key=qnum)).fillna("").astype(str).replace(r"\.0$", "", regex=True)

164 pipelines -> pipeline/


plan,default,front_02,front_03,front_04,front_05,knee,min_C,min_T,no_bf
query,,,,,,,,,
q2,29,26,26,26,26,26,26,26,24
q3,14,14,14,14,14,14,14,14,12
q4,9,8,8,8,8,8,8,8,8
q5,26,25,25,25,25,25,25,25,24
q7,23,23,23,23,23,23,23,23,22
q8,33,32,32,32,32,32,32,32,30
q9,25,25,25,25,25,25,25,25,24
q10,17,17,17,17,17,17,17,17,15
q11,19,18,,18,18,18,18,18,16


## Check against the pipelines that were run

In [8]:
RAN = os.path.expanduser("~/Sites/BLOOM-FaaS/benchmarks/plans-j2-validate")
if os.path.isdir(RAN):
    same, diff, missing = 0, [], []
    for p in W[W.plan != "no_bf"].path:
        ref = os.path.join(RAN, os.path.relpath(p, OUT))
        if not os.path.exists(ref):
            missing.append(ref)
        elif filecmp.cmp(p, ref, shallow=False):
            same += 1
        else:
            diff.append(p)
    print(f"identical to the pipelines run: {same}/{same + len(diff) + len(missing)}")
    print("different:", diff or "none"); print("not run:", missing or "none")
else:
    print(f"{RAN} not found, skipped")

identical to the pipelines run: 145/145
different: none
not run: none
